In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
from ast import literal_eval

df = pd.read_csv('data/matches_processed.csv')
df["Player"] = df["Player"].apply(literal_eval)

In [3]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

class WinPredictionDataset(Dataset):
    def __init__(self, players, result):
        self.players = players
        self.results = torch.tensor(result, dtype=torch.float32)
        
        # Flatten all tokens to build vocabulary
        all_tokens = [item for row in players for subsublist in row for item in subsublist]
        self.tokenizer = LabelEncoder()
        self.tokenizer.fit(all_tokens)  # Fit on all possible tokens
        
        # Pre-encode all data during init (more efficient)
        self.encoded_players = [
            [
                self.tokenizer.transform(subsublist) 
                for subsublist in row
            ] 
            for row in players
        ]
        
    def __len__(self):
        return len(self.players)

    def __getitem__(self, idx):
        return {
            "players": torch.tensor(self.encoded_players[idx], dtype=torch.long),  # Shape: [10, 5]
            "results": self.results[idx]  # Shape: [1]
        }

In [4]:
import torch
import torch.nn as nn

class WinPredictionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, encoder_dim=64):
        super().__init__()
        
        # 1. Embedding Layer 
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        
        # 2. 2D CNN (Spatial Feature Extraction)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3,3), padding=1),  # [batch, 32, 10, 5]
            nn.BatchNorm2d(16),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),  # [batch, 32, 5, 2]
        )
        
        # 3. Encoder (Processes CNN outputs)
        self.encoder = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=16*2,  # Flattened CNN channels
                nhead=8,
                dim_feedforward=encoder_dim,
                dropout=0.2,
            ),
            num_layers=1
        )
        
        # 4. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(16*2*5, 64),  # 5 players after pooling
            nn.Sigmoid(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: [batch_size, 10, 5]
        
        # 1. Embed tokens
        x = self.embedding(x)  # [batch, 10, 5, embed_dim]
        
        # 2. Prepare for CNN (average embeddings)
        x = x.mean(dim=-1, keepdim=True)  # [batch, 10, 5, 1]
        x = x.permute(0, 3, 1, 2)  # [batch, 1, 10, 5]
        
        # 3. 2D CNN
        cnn_out = self.cnn(x)  # [batch, 32, 5, 2]
        
        # 4. Prepare for encoder
        batch_size, channels, h, w = cnn_out.shape
        cnn_flat = cnn_out.reshape(batch_size, h, channels*w)  # [batch, 5, 64]
        
        # 5. Transformer encoder
        encoded = self.encoder(cnn_flat)  # [batch, 5, 64]
        
        # 6. Classifier
        out = self.classifier(encoded.reshape(batch_size, -1))
        return torch.sigmoid(out)

In [5]:
import plotly.graph_objects as go

def visualize_losses(train_losses, val_losses=None, title="Training and Validation Loss", xaxis_title="Epochs", yaxis_title="Binary Cross Entropy Loss"):
    fig = go.Figure()
        
    fig.add_trace(
        go.Scatter(
            x=list(range(1, len(train_losses) + 1)),
            y=train_losses,
            mode='lines',
            name='Train Loss',
            line=dict(color='blue'),
        )
    )
    if val_losses is not None:
        # Only add validation loss if provided
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(val_losses) + 1)),
                y=val_losses,
                mode='lines',
                name='Validation Loss',
                line=dict(color='orange'),
            )
        )
    
    fig.update_layout(
        title=title,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        template="plotly_white"
    )

    fig.show()

In [6]:
from sklearn.metrics import confusion_matrix, accuracy_score

def eval_model(model, val_loader, device):
    model.eval()
            
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            players = batch["players"].to(device)
            labels = batch["results"].cpu().numpy()
            
            outputs = model(players).cpu().numpy()
            
            preds = np.round(outputs)
            
            all_preds.extend(preds.flatten().tolist())
            all_labels.extend(labels.flatten().tolist())
            
    accuracy = accuracy_score(all_labels, all_preds)
    conf_matrix = confusion_matrix(all_labels, all_preds)  # Call the function from sklearn.metrics
    return accuracy, conf_matrix

In [8]:
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

dataset = WinPredictionDataset(df["Player"].values, df["Win"].to_numpy())

kf = KFold(n_splits=5, shuffle=True, random_state=42)

batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_epochs = 10

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"Fold {fold + 1}/{kf.n_splits}")
    
    trainset = torch.utils.data.Subset(dataset, train_idx)
    valset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)
    
    # Initialize model, optimizer, and loss function
    model = WinPredictionModel(vocab_size=dataset.tokenizer.classes_.size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()

    train_losses = []
    val_accuracies = []
    val_conf_matrices = []

    for epoch in range(num_epochs):
        train_loss = 0
        model.train()
        for batch in train_loader:
            players = batch["players"].to(device)
            results = batch["results"].to(device)
            
            optimizer.zero_grad()
            outputs = model(players)
            loss = criterion(outputs, results.unsqueeze(1))  # Ensure results are the same shape as outputs
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        average_train_loss = train_loss / len(train_loader)
        accuracy, conf_matrix = eval_model(model, val_loader, device)
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Average Train Loss: {average_train_loss}, Validation accuracy: {accuracy}")
        print(f"Confusion Matrix:\n{conf_matrix}")
        
        train_losses.append(average_train_loss)
        val_accuracies.append(accuracy)
        val_conf_matrices.append(conf_matrix)

    visualize_losses(train_losses)
    visualize_losses(val_accuracies, title="Validation Accuracy")

Using device: cuda
Fold 1/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



Epoch 1/10, Average Train Loss: 0.6914009017944336, Validation accuracy: 0.5155
Confusion Matrix:
[[460 516]
 [453 571]]
Epoch 2/10, Average Train Loss: 0.686108952999115, Validation accuracy: 0.5105
Confusion Matrix:
[[410 566]
 [413 611]]
Epoch 3/10, Average Train Loss: 0.6825084595680236, Validation accuracy: 0.5255
Confusion Matrix:
[[106 870]
 [ 79 945]]
Epoch 4/10, Average Train Loss: 0.677352391242981, Validation accuracy: 0.528
Confusion Matrix:
[[469 507]
 [437 587]]
Epoch 5/10, Average Train Loss: 0.6721331915855407, Validation accuracy: 0.548
Confusion Matrix:
[[362 614]
 [290 734]]
Epoch 6/10, Average Train Loss: 0.6686931166648865, Validation accuracy: 0.5445
Confusion Matrix:
[[477 499]
 [412 612]]
Epoch 7/10, Average Train Loss: 0.6670174479484559, Validation accuracy: 0.549
Confusion Matrix:
[[306 670]
 [232 792]]
Epoch 8/10, Average Train Loss: 0.6654748048782348, Validation accuracy: 0.546
Confusion Matrix:
[[550 426]
 [482 542]]
Epoch 9/10, Average Train Loss: 0.6630

Fold 2/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



...

c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



Epoch 1/10, Average Train Loss: 0.6922199144363403, Validation accuracy: 0.5305
Confusion Matrix:
[[117 821]
 [118 944]]
Epoch 2/10, Average Train Loss: 0.6892600111961364, Validation accuracy: 0.531
Confusion Matrix:
[[286 652]
 [286 776]]
Epoch 3/10, Average Train Loss: 0.6827577476501465, Validation accuracy: 0.5565
Confusion Matrix:
[[414 524]
 [363 699]]
Epoch 4/10, Average Train Loss: 0.6768999600410461, Validation accuracy: 0.556
Confusion Matrix:
[[666 272]
 [616 446]]
Epoch 5/10, Average Train Loss: 0.6722968091964722, Validation accuracy: 0.5515
Confusion Matrix:
[[604 334]
 [563 499]]
Epoch 6/10, Average Train Loss: 0.6706229753494263, Validation accuracy: 0.548
Confusion Matrix:
[[622 316]
 [588 474]]
Epoch 7/10, Average Train Loss: 0.6664167127609253, Validation accuracy: 0.563
Confusion Matrix:
[[504 434]
 [440 622]]
Epoch 8/10, Average Train Loss: 0.6655101914405823, Validation accuracy: 0.5665
Confusion Matrix:
[[418 520]
 [347 715]]
Epoch 9/10, Average Train Loss: 0.66

Fold 5/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



Epoch 1/10, Average Train Loss: 0.6924881629943848, Validation accuracy: 0.5345
Confusion Matrix:
[[ 96 826]
 [105 973]]
Epoch 2/10, Average Train Loss: 0.6903578939437867, Validation accuracy: 0.5305
Confusion Matrix:
[[548 374]
 [565 513]]
Epoch 3/10, Average Train Loss: 0.686206346988678, Validation accuracy: 0.539
Confusion Matrix:
[[322 600]
 [322 756]]
Epoch 4/10, Average Train Loss: 0.6840347652435302, Validation accuracy: 0.4855
Confusion Matrix:
[[710 212]
 [817 261]]
Epoch 5/10, Average Train Loss: 0.6825528454780578, Validation accuracy: 0.5325
Confusion Matrix:
[[269 653]
 [282 796]]
Epoch 6/10, Average Train Loss: 0.6790653815269471, Validation accuracy: 0.5185
Confusion Matrix:
[[437 485]
 [478 600]]
Epoch 7/10, Average Train Loss: 0.6737202138900756, Validation accuracy: 0.522
Confusion Matrix:
[[412 510]
 [446 632]]
Epoch 8/10, Average Train Loss: 0.6726606931686402, Validation accuracy: 0.5295
Confusion Matrix:
[[408 514]
 [427 651]]
Epoch 9/10, Average Train Loss: 0.6